In [1]:
import pandas as pd
import polars as pl
import os

In [2]:
INPUT_DIR = 'fromGoogleDrive/outputCompcor'

In [3]:
all_temp_dfs = []
for combination in os.listdir(f'./{INPUT_DIR}/ksc'):
    if os.path.isdir(f'./{INPUT_DIR}/ksc/{combination}'):
        combination_splits = combination.split('_')
        dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
        temp_df = pd.read_csv(f'./{INPUT_DIR}/ksc/{combination}/{combination}_ksc_metrics_measures.csv')
        temp_df = temp_df.groupby('metric').mean()
        temp_df['combination'] = f"{dataset1}_{dataset2}"
        temp_df['dataset1'] = dataset1
        temp_df['dataset2'] = dataset2
        temp_df['repetitions'] = repetitions
        all_temp_dfs.append(temp_df)

df = pd.concat(all_temp_dfs)

In [4]:
sorted_mean_df = df.drop(columns='Time').groupby('metric').mean(numeric_only=True)
sorted_median_df = df.drop(columns='Time').groupby('metric').median(numeric_only=True)
sorted_mean_df['Overall'] = sorted_mean_df.mean(axis=1).tolist()
sorted_median_df['Overall'] = sorted_median_df.median(axis=1).tolist()
sorted_mean_df['Algorithm'] = sorted_mean_df.index
sorted_median_df['Algorithm'] = sorted_median_df.index
print("----------------- Mean -----------------")
print(pl.from_pandas(sorted_mean_df.sort_values(by='Overall', ascending=False)))
print("----------------- Median -----------------")
print(pl.from_pandas(sorted_median_df.sort_values(by='Overall', ascending=False)))

----------------- Mean -----------------
shape: (10, 7)
┌──────────┬───────────────────┬──────────────┬──────────────┬───────────┬───────────┬─────────────┐
│ Accuracy ┆ Weighted Accuracy ┆ Monotonicity ┆ Separability ┆ Linearity ┆ Overall   ┆ Algorithm   │
│ ---      ┆ ---               ┆ ---          ┆ ---          ┆ ---       ┆ ---       ┆ ---         │
│ f64      ┆ f64               ┆ f64          ┆ f64          ┆ f64       ┆ f64       ┆ str         │
╞══════════╪═══════════════════╪══════════════╪══════════════╪═══════════╪═══════════╪═════════════╡
│ 0.964675 ┆ 0.945231          ┆ 0.938049     ┆ 0.890379     ┆ 0.941231  ┆ 0.935913  ┆ MAUVE       │
│ 0.950382 ┆ 0.926429          ┆ 0.927859     ┆ 0.881388     ┆ 0.945663  ┆ 0.926344  ┆ ZERO        │
│ 0.924228 ┆ 0.894078          ┆ 0.887612     ┆ 0.811251     ┆ 0.911131  ┆ 0.88566   ┆ TRADITIONAL │
│ 0.952058 ┆ 0.928856          ┆ 0.858755     ┆ 0.78361      ┆ 0.893581  ┆ 0.883372  ┆ FID         │
│ 0.846218 ┆ 0.793754          ┆ 0.

In [5]:
datasets = [
    'clinicalDialogueSummarizations',
    # 'dementiaAudio',
    'medicalAbstracts',
    'syntheticCareHomeNurseNotes',
    'simSUM'
]

filtered = df[
    df['dataset1'].isin(datasets) | df['dataset2'].isin(datasets)
]

sorted_mean_df = filtered.drop(columns='Time').groupby('metric').mean(numeric_only=True)
sorted_median_df = filtered.drop(columns='Time').groupby('metric').median(numeric_only=True)
sorted_mean_df['Overall'] = sorted_mean_df.mean(axis=1).tolist()
sorted_median_df['Overall'] = sorted_median_df.median(axis=1).tolist()
sorted_mean_df['Algorithm'] = sorted_mean_df.index
sorted_median_df['Algorithm'] = sorted_median_df.index
print("----------------- Mean -----------------")
print(pl.from_pandas(sorted_mean_df.sort_values(by='Overall', ascending=False)))
print("----------------- Median -----------------")
print(pl.from_pandas(sorted_median_df.sort_values(by='Overall', ascending=False)))

----------------- Mean -----------------
shape: (10, 7)
┌──────────┬───────────────────┬──────────────┬──────────────┬───────────┬───────────┬─────────────┐
│ Accuracy ┆ Weighted Accuracy ┆ Monotonicity ┆ Separability ┆ Linearity ┆ Overall   ┆ Algorithm   │
│ ---      ┆ ---               ┆ ---          ┆ ---          ┆ ---       ┆ ---       ┆ ---         │
│ f64      ┆ f64               ┆ f64          ┆ f64          ┆ f64       ┆ f64       ┆ str         │
╞══════════╪═══════════════════╪══════════════╪══════════════╪═══════════╪═══════════╪═════════════╡
│ 0.970685 ┆ 0.955114          ┆ 0.944532     ┆ 0.906235     ┆ 0.95893   ┆ 0.947099  ┆ ZERO        │
│ 0.979316 ┆ 0.965863          ┆ 0.94451      ┆ 0.892452     ┆ 0.944088  ┆ 0.945246  ┆ MAUVE       │
│ 0.959467 ┆ 0.937506          ┆ 0.929149     ┆ 0.878609     ┆ 0.946768  ┆ 0.9303    ┆ TRADITIONAL │
│ 0.974206 ┆ 0.959023          ┆ 0.893809     ┆ 0.843993     ┆ 0.921677  ┆ 0.918541  ┆ FID         │
│ 0.874116 ┆ 0.826208          ┆ 0.